In [ ]:
# 1. Mount Google Drive to save the model later safely
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Install required libraries
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers streamlit

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 2. Humara updated private data (Knowledge Base)
my_data = """Hello! I'm Muhammad Awais, a specialized WordPress Developer and Full-Stack Web Designer, SaaS developer, AI/ML Engineer with a passion for building high-performance digital solutions. With over 1.5+ years of industry experience, I transform complex ideas into intuitive, fast-loading, and visually stunning websites as well as SaaS and AI products."""

with open("devxyn_knowledge.txt", "w") as f:
    f.write(my_data)

# 3. Load & Split Text (Break the document into small chunks)
loader = TextLoader("devxyn_knowledge.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = text_splitter.split_documents(documents)

# 4. Create Vector Database (FAISS - AI memory)
print(" AI understanding the data (Making Vector DB)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(docs, embeddings)


In [ ]:
# 5. small Test (Similarity Search)
query = "What does Awais specialize in?"
result = vector_store.similarity_search(query)
print("\n Setup Complete! AI Memory test result:")
print(" AI Found:", result[0].page_content)

In [ ]:
%%writefile app.py
import streamlit as st
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Page Setup
st.title(" Devxyn AI Chatbot (RAG)")
st.caption("Ask me anything about Muhammad Awais or Devxyn!")

# 2. AI Memory Loading
@st.cache_resource
def load_knowledge():
    loader = TextLoader("devxyn_knowledge.txt")
    documents = loader.load()
    text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20)
    docs = text_splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(docs, embeddings)
    return vector_store

db = load_knowledge()

# 3. Chat History (Context Memory)
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# 4. User Input and AI Response
user_query = st.chat_input("Ask a question...")

if user_query:
    # Show message of user
    with st.chat_message("user"):
        st.markdown(user_query)
    st.session_state.messages.append({"role": "user", "content": user_query})

    # Find from Vector DB
    results = db.similarity_search(user_query)
    best_answer = results[0].page_content if results else "Sorry, I don't know about that."

    # Show response of AI
    with st.chat_message("assistant"):
        st.markdown(f"**Found Info:** {best_answer}")
    st.session_state.messages.append({"role": "assistant", "content": f"**Found Info:** {best_answer}"})

# ***Testing the bot***

In [ ]:
import gradio as gr
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print(" Starting AI Engine...")

# 1. Load AI Memory (Vector DB)
loader = TextLoader("devxyn_knowledge.txt")
docs = CharacterTextSplitter(chunk_size=100, chunk_overlap=20).split_documents(loader.load())
db = FAISS.from_documents(docs, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

# 2. Chatbot Logic
def chat_with_rag(message, history):
    # Find the user's question in database
    results = db.similarity_search(message)
    answer = results[0].page_content if results else "Sorry, I don't have information on that yet."
    return f"**Tymelyte AI Engine:** {answer}"

# 3. Gradio Chat UI
interface = gr.ChatInterface(
    fn=chat_with_rag,
    title=" Devxyn RAG Chatbot Test",
    description="Ask anything about Awais, Devxyn, or your skills!"
)

# 4. Launch!
interface.launch(share=True)

In [ ]:
import gradio as gr
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print(" Updating AI Engine with better memory...")

# 1. Data ko lines mein tod diya taa ke AI alag alag baatein yaad rakhay
my_data = """Muhammad Awais is a specialized WordPress Developer and Full-Stack Web Designer.
He is also a SaaS developer and an AI/ML Engineer.
Awais has a passion for building high-performance digital solutions.
With over 1.5+ years of industry experience, he transforms complex ideas into intuitive, fast-loading, and visually stunning websites.
He also builds advanced SaaS and AI products.
He is the founder of Devxyn, a digital agency in Lahore.
Awais is currently studying at UCP and his CGPA is 3.43.
He is building an autonomous AI companion named Tymelyte.
He frequently collaborates with his friend Tahire."""

with open("devxyn_knowledge_v2.txt", "w") as f:
    f.write(my_data)

# 2. Use Recursive Splitter (Yeh sentences ko theek se todta hai)
loader = TextLoader("devxyn_knowledge_v2.txt")
# chunk_size chota rakha hai aur chunk_overlap bhi, taa ke specific answers aayein
docs = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20).split_documents(loader.load())
db = FAISS.from_documents(docs, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

# 3. Chatbot Logic
def chat_with_rag(message, history):
    # k=1 ka matlab hai sirf 1 sab se best matching line nikal kar do
    results = db.similarity_search(message, k=1)
    answer = results[0].page_content if results else "Sorry, data not found."
    return f"**Tymelyte AI:** {answer}"

# 4. UI Launch
interface = gr.ChatInterface(
    fn=chat_with_rag,
    title=" Tymelyte RAG Engine (Fixed)",
    description="Ask specific questions like: 'What is his CGPA?', 'Who is Tahir?', 'What is Tymelyte?'"
)
interface.launch(share=True)